# 🏦 Transaction Category Classifier
## Finsight ML — Training Notebook (Google Colab)

**Tujuan:** Melatih model BiLSTM untuk mengklasifikasikan deskripsi transaksi keuangan Indonesia ke 7 kategori pengeluaran.

**Pipeline posisi model ini:**
```
Gambar Struk
    → PaddleOCR
    → ReceiptLineClassifier (12 kelas per baris)
    → ReceiptExtractor → {store, date, items, total}
    → TransactionCategoryClassifier  ← MODEL INI
    → {category, confidence}
```

**Input model:** `deskripsi` (string) — nama item atau nama toko dari hasil OCR  
**Output model:** `category` (7 kelas) + `confidence` (float)

---
**7 Kategori:** `makanan` · `belanja` · `transportasi` · `tagihan` · `kesehatan` · `hiburan` · `lainnya`

## ⚙️ Setup

In [ ]:
# Install dependensi (jalankan sekali)
!pip install -q tensorflow scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import os
import json
import random
import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, Model

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {len(tf.config.list_physical_devices("GPU")) > 0}')

# Seed untuk reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# ── Konfigurasi ──────────────────────────────────────────────
CATEGORIES   = ['makanan', 'belanja', 'hiburan', 'transportasi', 'tagihan', 'kesehatan', 'lainnya']
NUM_CLASSES  = len(CATEGORIES)

# Hyperparameter model
MAX_LEN      = 30    # panjang token maksimum per teks
VOCAB_SIZE   = 5000  # ukuran vocabulary
EMBED_DIM    = 64
LSTM_UNITS   = 128
DROPOUT      = 0.3

# Training
EPOCHS       = 50
BATCH_SIZE   = 64
LR           = 1e-3

# Augmentasi: jumlah sampel sintetik tambahan per kategori
AUGMENT_PER_CAT = 1000

# Output dir (Google Drive atau Colab local)
OUTPUT_DIR = Path('/content/drive/MyDrive/finsight-ml/transaction_classifier')
# Kalau tidak pakai Drive, ganti ke:
# OUTPUT_DIR = Path('/content/transaction_classifier')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Kategori    : {CATEGORIES}')
print(f'Output dir  : {OUTPUT_DIR}')

## 📂 Mount Google Drive (Opsional)
Untuk menyimpan model langsung ke Drive agar tidak hilang saat sesi Colab berakhir.

In [ ]:
# Jalankan cell ini jika ingin menyimpan ke Google Drive
from google.colab import drive
drive.mount('/content/drive')

## 📦 Step 1 — Load Dataset

Upload 3 file CSV dari folder `category_classifier/data/`:
- `dataset_eda_final.csv` — 3,578 baris, 7 kategori (dataset utama)
- `df_synthetic.csv` — 2,100 baris, 7 kategori (sintetik Indonesia, balanced)
- `df_real.csv` — 74 baris, 7 kategori (data real berlabel)

In [ ]:
# Pilih salah satu: upload manual atau letakkan file di /content/
from google.colab import files

print('Upload 3 file CSV: dataset_eda_final.csv, df_synthetic.csv, df_real.csv')
uploaded = files.upload()  # akan muncul dialog upload

# Setelah upload, file ada di /content/
DATA_DIR = Path('/content')
print('\nFile terupload:')
for f in uploaded.keys():
    print(f'  {f}')

In [ ]:
# ── Load & merge semua dataset ───────────────────────────────
def load_csv(path, label_col='category', text_col='deskripsi'):
    df = pd.read_csv(path)
    df = df[[text_col, label_col]].rename(columns={text_col: 'deskripsi', label_col: 'category'})
    df = df.dropna()
    df['category'] = df['category'].str.lower().str.strip()
    return df

# Mapping kategori lama → 7 kategori final
CATEGORY_MAP = {
    'kesehatan dan perawatan diri': 'kesehatan',
    'travel'                      : 'transportasi',
    'sosial'                      : 'hiburan',
    'pendidikan'                  : 'lainnya',
    'kebugaran'                   : 'kesehatan',
}

dfs = []
for fname in ['dataset_eda_final.csv', 'df_synthetic.csv', 'df_real.csv']:
    path = DATA_DIR / fname
    if path.exists():
        # df_real.csv punya kolom 'label' numerik, skip itu
        df = load_csv(path)
        df['source'] = fname.replace('.csv', '')
        dfs.append(df)
        print(f'  Loaded {fname:30s}: {len(df):>5,} baris')
    else:
        print(f'  ⚠️  {fname} tidak ditemukan, skip.')

all_df = pd.concat(dfs, ignore_index=True)

# Apply category mapping
all_df['category'] = all_df['category'].replace(CATEGORY_MAP)

# Filter hanya 7 kategori yang valid
all_df = all_df[all_df['category'].isin(CATEGORIES)].copy()

# Drop duplicates berdasarkan teks
all_df = all_df.drop_duplicates(subset=['deskripsi']).reset_index(drop=True)

print(f'\nTotal setelah merge & deduplicate: {len(all_df):,} baris')
print(f'\nDistribusi kategori:')
print(all_df['category'].value_counts().to_string())

## 📊 Step 2 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribusi kategori
cat_counts = all_df['category'].value_counts()
axes[0].barh(cat_counts.index, cat_counts.values, color='#1E88E5')
axes[0].set_title('Distribusi Kategori', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Jumlah Data')
for i, v in enumerate(cat_counts.values):
    axes[0].text(v + 5, i, str(v), va='center', fontsize=10)

# Distribusi panjang teks
all_df['word_count'] = all_df['deskripsi'].str.split().str.len()
axes[1].hist(all_df['word_count'], bins=20, color='#42A5F5', edgecolor='white')
axes[1].set_title('Distribusi Jumlah Kata per Deskripsi', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Jumlah Kata')
axes[1].set_ylabel('Frekuensi')
axes[1].axvline(all_df['word_count'].mean(), color='red', linestyle='--',
                label=f'Mean: {all_df["word_count"].mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nRata-rata jumlah kata : {all_df["word_count"].mean():.1f}')
print(f'Max jumlah kata       : {all_df["word_count"].max()}')
print(f'Imbalance ratio       : {cat_counts.max() / cat_counts.min():.1f}x')

In [ ]:
# Sample per kategori
print('=== Sample deskripsi per kategori ===')
for cat in CATEGORIES:
    samples = all_df[all_df['category'] == cat]['deskripsi'].head(5).tolist()
    print(f'\n[{cat.upper()}]')
    for s in samples:
        print(f'  · {s}')

## 🔧 Step 3 — Text Preprocessing

In [ ]:
def clean_text(text: str) -> str:
    """Normalize teks deskripsi transaksi."""
    if not isinstance(text, str):
        return ''
    text = text.lower().strip()
    # Hapus karakter khusus kecuali spasi
    text = re.sub(r'[^\w\s]', ' ', text)
    # Hapus angka berdiri sendiri (bukan bagian kata)
    text = re.sub(r'\b\d+\b', '', text)
    # Normalisasi whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning
all_df['text_clean'] = all_df['deskripsi'].apply(clean_text)

# Hapus baris yang jadi kosong setelah cleaning
all_df = all_df[all_df['text_clean'].str.len() > 0].reset_index(drop=True)

# Preview
print('Contoh hasil cleaning:')
sample = all_df[['deskripsi', 'text_clean', 'category']].sample(8, random_state=42)
print(sample.to_string(index=False))

## 🔄 Step 4 — Data Augmentasi

Augmentasi dilakukan dengan men-generate deskripsi sintetik baru menggunakan:
- **Merchant prefixes** lokal Indonesia per kategori
- **Template kalimat** yang variatif
- **Typo** acak (15% probabilitas)
- **Singkatan** umum (20% probabilitas)

In [ ]:
MERCHANT_PREFIXES = {
    'makanan': [
        'warung', 'rm', 'resto', 'kedai', 'mie', 'nasi', 'bakso', 'soto',
        'warteg', 'kantin', 'cafe', 'kopi', 'boba', 'pizza', 'burger',
        'ayam', 'bebek', 'seafood', 'padang', 'sunda', 'jawa', 'mcd',
        'kfc', 'hokben', 'indomaret', 'alfamart', 'grab food', 'gofood',
        'shopee food', 'starbucks', 'jco', 'dunkin', 'chatime', 'xing fu tang'
    ],
    'belanja': [
        'shopee', 'tokopedia', 'lazada', 'blibli', 'tiktok shop', 'bukalapak',
        'indomaret', 'alfamart', 'hypermart', 'carrefour', 'giant', 'lottemart',
        'ace hardware', 'ikea', 'h&m', 'zara', 'uniqlo', 'miniso', 'daiso',
        'guardian', 'watson', 'century', 'apotek', 'toko', 'pasar'
    ],
    'transportasi': [
        'grab', 'gojek', 'maxim', 'indriver', 'ojol', 'ojek', 'taxi',
        'blue bird', 'express', 'transjakarta', 'mrt', 'lrt', 'kereta',
        'kai', 'damri', 'bus', 'angkot', 'bensin', 'pertamina', 'shell',
        'spbu', 'parkir', 'tol', 'jasa marga'
    ],
    'tagihan': [
        'pln', 'listrik', 'pdam', 'air', 'telkom', 'indihome', 'firstmedia',
        'biznet', 'myrepublic', 'telkomsel', 'xl', 'indosat', 'tri', 'smartfren',
        'bpjs', 'cicilan', 'kpr', 'angsuran', 'iuran', 'sewa', 'kos', 'kontrakan'
    ],
    'kesehatan': [
        'rs', 'rumah sakit', 'klinik', 'puskesmas', 'dokter', 'apotek',
        'kimia farma', 'guardian', 'century', 'k24', 'halodoc', 'alodokter',
        'good doctor', 'lab', 'laboratorium', 'radiologi', 'optik', 'dental'
    ],
    'hiburan': [
        'netflix', 'spotify', 'youtube', 'disney+', 'vidio', 'viu', 'mola',
        'cgv', 'cinepolis', 'xxi', 'bioskop', 'karaoke', 'inul vizta',
        'timezone', 'amazone', 'wahana', 'taman', 'museum', 'konser',
        'steam', 'playstation', 'xbox', 'nintendo', 'mobile legend', 'game'
    ],
    'lainnya': [
        'atm', 'transfer', 'admin', 'biaya', 'fee', 'cashback', 'refund',
        'bca', 'bni', 'bri', 'mandiri', 'cimb', 'danamon', 'ovo', 'dana',
        'gopay', 'shopeepay', 'linkaja', 'qris', 'top up', 'pulsa'
    ]
}

TEMPLATES = {
    'makanan'      : ['{p}', '{p} {loc}', 'beli {p}', 'makan {p}', 'order {p}',
                      '{p} delivery', 'jajan {p}', 'sarapan', 'makan siang {p}',
                      'makan malam {p}', 'snack {p}', 'kopi {p}'],
    'belanja'      : ['beli di {p}', '{p}', 'belanja {p}', 'pembelian {p}',
                      'order {p}', '{p} online', 'beli {item}', 'belanja {item}'],
    'transportasi' : ['{p}', 'naik {p}', 'bayar {p}', '{p} online', 'isi {p}',
                      'top up {p}', 'parkir {p}', 'tol {p}', 'tiket {p}'],
    'tagihan'      : ['bayar {p}', 'tagihan {p}', '{p}', 'cicilan {p}',
                      'iuran {p}', 'sewa {p}', 'pembayaran {p}', 'angsuran {p}'],
    'kesehatan'    : ['{p}', 'berobat {p}', 'beli obat {p}', 'konsultasi {p}',
                      'cek {p}', 'periksa {p}', 'bayar {p}', 'kunjungan {p}'],
    'hiburan'      : ['{p}', 'langganan {p}', 'subscribe {p}', 'nonton {p}',
                      'main {p}', 'beli {p}', 'top up {p}', 'bayar {p}'],
    'lainnya'      : ['{p}', 'biaya {p}', 'admin {p}', 'transfer {p}',
                      'top up {p}', 'bayar {p}', 'fee {p}', 'cashback {p}']
}

ITEMS = ['baju', 'celana', 'sepatu', 'tas', 'elektronik', 'hp', 'laptop',
         'buku', 'alat tulis', 'perabot', 'mainan', 'kosmetik', 'skincare']
LOCS  = ['terdekat', 'mall', 'online', 'depan kantor', 'pinggir jalan']

TYPO_MAP = {
    'a': ['4', '@'], 'e': ['3'], 'i': ['1', 'y'], 'o': ['0'], 's': ['5', 'z'],
    'makan': ['mkn', 'mkan'], 'belanja': ['blnja', 'blj'], 'bayar': ['byr'],
    'transfer': ['trf', 'tf'], 'tagihan': ['tghn'], 'listrik': ['lstrik'],
    'bensin': ['bnsn'],
}

ABBREVIATIONS = {
    'rumah sakit': 'rs', 'restoran': 'resto', 'warung makan': 'warmak',
    'makanan': 'mkn', 'transportasi': 'transport', 'pembayaran': 'bayar',
    'tagihan': 'tgh', 'listrik': 'lstrik', 'internet': 'inet',
}


def apply_typo(text: str, prob: float = 0.15) -> str:
    words = text.split()
    result = []
    for word in words:
        if random.random() < prob and len(word) > 3:
            if word in TYPO_MAP:
                word = random.choice(TYPO_MAP[word])
            else:
                chars = list(word)
                idx = random.randint(0, len(chars) - 1)
                c = chars[idx].lower()
                if c in TYPO_MAP:
                    chars[idx] = random.choice(TYPO_MAP[c])
                word = ''.join(chars)
        result.append(word)
    return ' '.join(result)


def apply_abbreviation(text: str, prob: float = 0.2) -> str:
    for full, abbr in ABBREVIATIONS.items():
        if full in text and random.random() < prob:
            text = text.replace(full, abbr)
    return text


def generate_synthetic(category: str, n: int) -> list:
    samples = []
    prefixes = MERCHANT_PREFIXES.get(category, MERCHANT_PREFIXES['lainnya'])
    tmpl_list = TEMPLATES.get(category, TEMPLATES['lainnya'])

    for _ in range(n):
        prefix = random.choice(prefixes)
        tmpl   = random.choice(tmpl_list)
        text   = tmpl.format(p=prefix, item=random.choice(ITEMS), loc=random.choice(LOCS))

        aug = random.random()
        if   aug < 0.15: text = apply_typo(text)
        elif aug < 0.30: text = apply_abbreviation(text)
        elif aug < 0.40: text = text.upper()
        elif aug < 0.50: text = text.title()

        samples.append({'deskripsi': text, 'category': category, 'source': 'augmented'})

    return samples


print('Fungsi augmentasi siap.')

In [ ]:
print(f'Membuat {AUGMENT_PER_CAT} sampel sintetik per kategori...')

synthetic_rows = []
for cat in CATEGORIES:
    samples = generate_synthetic(cat, AUGMENT_PER_CAT)
    synthetic_rows.extend(samples)

synth_df = pd.DataFrame(synthetic_rows)
synth_df['text_clean'] = synth_df['deskripsi'].apply(clean_text)

# Gabungkan dengan data asli
combined_df = pd.concat(
    [all_df[['deskripsi', 'text_clean', 'category']],
     synth_df[['deskripsi', 'text_clean', 'category']]],
    ignore_index=True
)
combined_df = combined_df[combined_df['text_clean'].str.len() > 0]
combined_df = combined_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'\nDataset setelah augmentasi: {len(combined_df):,} baris')
print('\nDistribusi akhir:')
print(combined_df['category'].value_counts().to_string())

## 🔤 Step 5 — Tokenisasi & Encoding

In [ ]:
# ── Tokenizer ────────────────────────────────────────────────
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>', lower=True)
tokenizer.fit_on_texts(combined_df['text_clean'])

sequences = tokenizer.texts_to_sequences(combined_df['text_clean'])
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

# ── Label encoding ───────────────────────────────────────────
label2idx = {cat: i for i, cat in enumerate(CATEGORIES)}
idx2label = {i: cat for cat, i in label2idx.items()}

y_int  = combined_df['category'].map(label2idx).values
y_ohe  = tf.keras.utils.to_categorical(y_int, num_classes=NUM_CLASSES)

print(f'Vocab size actual   : {len(tokenizer.word_index):,}')
print(f'X shape             : {X.shape}')
print(f'y shape             : {y_ohe.shape}')
print(f'\nContoh tokenisasi:')
sample_texts = ['warung nasi padang', 'bayar tagihan listrik pln', 'grab car ke kantor']
for t in sample_texts:
    seq = tokenizer.texts_to_sequences([clean_text(t)])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding='post')
    print(f'  "{t}"')
    print(f'  → {pad[0][:10]} ...')

In [ ]:
# ── Train / Val / Test split ─────────────────────────────────
X_train, X_test, y_train, y_test, y_int_train, y_int_test = train_test_split(
    X, y_ohe, y_int, test_size=0.15, random_state=SEED, stratify=y_int
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=SEED
)

print(f'Train : {len(X_train):,}')
print(f'Val   : {len(X_val):,}')
print(f'Test  : {len(X_test):,}')

## 🧠 Step 6 — Arsitektur Model

**BiLSTM Text Classifier**
```
Input (max_len=30 token)
  → Embedding(vocab=5000, dim=64)
  → SpatialDropout1D(0.2)
  → BiLSTM(128, return_sequences=True)
  → BiLSTM(64)
  → Dense(128, relu) → Dropout(0.3)
  → Dense(64, relu)  → Dropout(0.3)
  → Dense(7, softmax)
```

In [ ]:
def build_model(vocab_size: int, num_classes: int, max_len: int,
                embed_dim: int = 64, lstm_units: int = 128,
                dropout: float = 0.3) -> tf.keras.Model:

    inputs = tf.keras.Input(shape=(max_len,), name='text_input')

    x = layers.Embedding(vocab_size, embed_dim, mask_zero=True, name='embedding')(inputs)
    x = layers.SpatialDropout1D(0.2, name='spatial_dropout')(x)

    x = layers.Bidirectional(
        layers.LSTM(lstm_units, return_sequences=True, dropout=dropout), name='bilstm_1'
    )(x)
    x = layers.Bidirectional(
        layers.LSTM(lstm_units // 2, dropout=dropout), name='bilstm_2'
    )(x)

    x = layers.Dense(128, activation='relu', name='dense_1')(x)
    x = layers.Dropout(dropout, name='dropout_1')(x)
    x = layers.Dense(64, activation='relu', name='dense_2')(x)
    x = layers.Dropout(dropout, name='dropout_2')(x)

    outputs = layers.Dense(num_classes, activation='softmax', name='classifier')(x)

    return tf.keras.Model(inputs, outputs, name='TransactionCategoryClassifier')


model = build_model(VOCAB_SIZE, NUM_CLASSES, MAX_LEN, EMBED_DIM, LSTM_UNITS, DROPOUT)
model.summary()

## 🚀 Step 7 — Training

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=7,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3,
        min_lr=1e-6, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        str(OUTPUT_DIR / 'best_model.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=0
    ),
    tf.keras.callbacks.CSVLogger(
        str(OUTPUT_DIR / 'training_log.csv')
    ),
]

print(f'Training dimulai — {EPOCHS} epochs max, early stopping patience=7')
print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}\n')

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train', color='#1E88E5')
axes[0].plot(history.history['val_accuracy'], label='Val',   color='#FB8C00')
axes[0].set_title('Accuracy per Epoch', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train', color='#1E88E5')
axes[1].plot(history.history['val_loss'], label='Val',   color='#FB8C00')
axes[1].set_title('Loss per Epoch', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_history.png', dpi=150, bbox_inches='tight')
plt.show()

best_val_acc = max(history.history['val_accuracy'])
print(f'Best val accuracy: {best_val_acc*100:.2f}%')

## 📊 Step 8 — Evaluasi

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy : {test_acc*100:.2f}%')
print(f'Test Loss     : {test_loss:.4f}')

y_pred     = model.predict(X_test, verbose=0)
y_pred_cls = np.argmax(y_pred, axis=1)
y_true_cls = np.argmax(y_test, axis=1)

print('\nClassification Report:')
print(classification_report(
    y_true_cls, y_pred_cls,
    target_names=CATEGORIES,
    digits=3
))

In [ ]:
cm = confusion_matrix(y_true_cls, y_pred_cls)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CATEGORIES, yticklabels=CATEGORIES, ax=axes[0])
axes[0].set_title('Confusion Matrix (Raw)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CATEGORIES, yticklabels=CATEGORIES, ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 💾 Step 9 — Simpan Artifacts

Semua artifacts disimpan ke `OUTPUT_DIR` (Google Drive atau Colab local).  
Download dan letakkan di `finsight-ml/models/transaction_classifier/`.

In [ ]:
# 1. Model weights
# best_model.keras sudah disimpan oleh ModelCheckpoint callback

# 2. Tokenizer
with open(OUTPUT_DIR / 'tokenizer.json', 'w', encoding='utf-8') as f:
    f.write(tokenizer.to_json())

# 3. Label mapping
with open(OUTPUT_DIR / 'label_mapping.json', 'w', encoding='utf-8') as f:
    json.dump({'label2idx': label2idx, 'idx2label': idx2label}, f, indent=2)

# 4. Config
config = {
    'max_len'              : MAX_LEN,
    'vocab_size'           : VOCAB_SIZE,
    'embed_dim'            : EMBED_DIM,
    'lstm_units'           : LSTM_UNITS,
    'dropout'              : DROPOUT,
    'categories'           : CATEGORIES,
    'test_accuracy'        : float(test_acc),
    'best_val_accuracy'    : float(best_val_acc),
    'total_train_samples'  : int(len(X_train)),
    'augmented_samples'    : int(len(synth_df)),
    'epochs_trained'       : int(len(history.history['accuracy'])),
}
with open(OUTPUT_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)

# 5. Training history
hist_data = {k: [float(v) for v in vals] for k, vals in history.history.items()}
with open(OUTPUT_DIR / 'training_history.json', 'w') as f:
    json.dump(hist_data, f, indent=2)

print('✅ Artifacts tersimpan:')
for f in sorted(OUTPUT_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:35s} {size_kb:>8.1f} KB')

In [ ]:
# Download artifacts ke komputer lokal (jika tidak pakai Drive)
from google.colab import files

for fname in ['best_model.keras', 'tokenizer.json', 'label_mapping.json', 'config.json']:
    fpath = OUTPUT_DIR / fname
    if fpath.exists():
        files.download(str(fpath))
        print(f'  Downloaded: {fname}')

## 🧪 Step 10 — Inference Test

Simulasi bagaimana model akan dipanggil di FastAPI setelah output dari `ReceiptExtractor`.

In [ ]:
def predict_category(deskripsi: str) -> dict:
    """Inference satu teks deskripsi."""
    cleaned = clean_text(deskripsi)
    seq = tokenizer.texts_to_sequences([cleaned])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding='post')
    probs = model.predict(pad, verbose=0)[0]
    idx   = int(np.argmax(probs))
    return {
        'deskripsi' : deskripsi,
        'category'  : idx2label[idx],
        'confidence': round(float(probs[idx]), 4),
        'all_probs' : {idx2label[i]: round(float(p), 4) for i, p in enumerate(probs)}
    }


def predict_from_ocr(store: str, items: list) -> dict:
    """Simulasi input dari output ReceiptExtractor."""
    # Gabungkan store + item names menjadi satu deskripsi
    parts = [store] + [item.get('name', '') for item in items]
    deskripsi = ' '.join(p for p in parts if p).strip()
    return predict_category(deskripsi)


print('Fungsi inference siap.')

In [ ]:
print('=== Test 1: Teks deskripsi biasa ===')
test_cases = [
    'makan siang warteg',
    'grab car ke kantor',
    'bayar tagihan listrik pln',
    'netflix subscription',
    'beli baju di shopee',
    'kunjungan dokter umum',
    'transfer bca',
    'gofood indomie goreng',
    'bensin pertamina',
    'bayar cicilan kpr'
]
for text in test_cases:
    r = predict_category(text)
    print(f"  '{r['deskripsi']:35s}' → {r['category']:15s} ({r['confidence']*100:.1f}%)")

In [ ]:
print('=== Test 2: Dari output ReceiptExtractor (simulasi) ===')

ocr_outputs = [
    {
        'store': 'Indomaret',
        'items': [{'name': 'Indomie Goreng', 'qty': 2, 'price': 3500},
                  {'name': 'Susu Ultra', 'qty': 1, 'price': 8000}],
        'total': 15000
    },
    {
        'store': 'Kopi Nako Summarecon',
        'items': [{'name': 'Iced Matcha Latte', 'qty': 1, 'price': 29000},
                  {'name': 'Kahlua Kopi', 'qty': 1, 'price': 27000}],
        'total': 56000
    },
    {
        'store': 'Grab',
        'items': [{'name': 'GrabCar', 'qty': 1, 'price': 35000}],
        'total': 35000
    },
    {
        'store': 'PLN Mobile',
        'items': [{'name': 'Token Listrik', 'qty': 1, 'price': 100000}],
        'total': 100000
    },
    {
        'store': 'Kimia Farma',
        'items': [{'name': 'Paracetamol', 'qty': 1, 'price': 8000},
                  {'name': 'Vitamin C', 'qty': 1, 'price': 15000}],
        'total': 23000
    },
]

for ocr in ocr_outputs:
    r = predict_from_ocr(ocr['store'], ocr['items'])
    item_names = ', '.join(i['name'] for i in ocr['items'])
    print(f"  Store: {ocr['store']:25s} | Items: {item_names:40s}")
    print(f"  → deskripsi: '{r['deskripsi']}'")
    print(f"  → category : {r['category']} ({r['confidence']*100:.1f}%)")
    print()

## ✅ Summary

| Artifact | Deskripsi | Dipakai di |
|---|---|---|
| `best_model.keras` | Model BiLSTM trained | FastAPI inference |
| `tokenizer.json` | Keras Tokenizer config | Preprocessing teks |
| `label_mapping.json` | `{label2idx, idx2label}` | Decode output model |
| `config.json` | Hyperparameter & metadata | Validasi & logging |
| `training_log.csv` | Loss & accuracy per epoch | Analisis training |
| `training_history.png` | Plot training curve | Dokumentasi |
| `confusion_matrix.png` | Confusion matrix test set | Analisis error |

**Langkah selanjutnya:**
1. Download semua artifacts dari `OUTPUT_DIR`
2. Letakkan di `finsight-ml/models/transaction_classifier/`
3. Implementasi `src/transaction_classifier.py` di finsight-ml untuk load & inference
4. Integrasi ke `web/api_v2.py` setelah ReceiptExtractor selesai